# 01-04 Softmax 函数公式推导

Softmax 常用于多分类任务的输出层。它的核心作用是：把一组任意实数分数转换成一组概率，并且所有概率之和为 $1$。

## 1. 为什么需要 Softmax

多分类模型通常先输出 $K$ 个原始分数：

$$
\mathbf{z}=[z_1,z_2,\dots,z_K]
$$

这些分数也叫 logits。它们不是概率，因为可能为负，也不一定相加为 $1$。

我们希望得到一组概率：

$$
\mathbf{p}=[p_1,p_2,\dots,p_K]
$$

并满足：

$$
0<p_i<1
$$

$$
\sum_{i=1}^{K}p_i=1
$$

Softmax 就是把 logits 转成概率分布的方法。

## 2. 从二分类 Sigmoid 到多分类 Softmax

二分类时，我们可以用 Sigmoid 输出类别 $1$ 的概率：

$$
p=\frac{1}{1+e^{-z}}
$$

也可以把二分类看成两个类别分数 $z_0$ 和 $z_1$ 的竞争：

$$
p_1=\frac{e^{z_1}}{e^{z_0}+e^{z_1}}
$$

如果令 $z=z_1-z_0$，那么：

$$
p_1=\frac{e^{z_1}}{e^{z_0}+e^{z_1}}
$$

分子分母同时除以 $e^{z_1}$：

$$
p_1=\frac{1}{e^{z_0-z_1}+1}
$$

$$
p_1=\frac{1}{1+e^{-(z_1-z_0)}}
$$

$$
p_1=\sigma(z_1-z_0)
$$

所以 Softmax 可以看成 Sigmoid 在多分类场景下的推广。

## 3. Softmax 公式怎么来

我们需要把每个类别分数 $z_i$ 变成正数。指数函数正好满足：

$$
e^{z_i}>0
$$

先得到每个类别的非归一化权重：

$$
s_i=e^{z_i}
$$

为了让所有类别的概率加起来等于 $1$，用所有权重的总和进行归一化：

$$
p_i=\frac{s_i}{\sum_{j=1}^{K}s_j}
$$

代入 $s_i=e^{z_i}$：

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

这就是 Softmax：

$$
\operatorname{softmax}(z_i)=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

## 4. 为什么输出一定是概率分布

首先，因为指数函数恒正：

$$
e^{z_i}>0
$$

所以：

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}>0
$$

其次，所有概率求和：

$$
\sum_{i=1}^{K}p_i=\sum_{i=1}^{K}\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

因为分母和 $i$ 无关，可以提出：

$$
\sum_{i=1}^{K}p_i=\frac{\sum_{i=1}^{K}e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

分子和分母是同一个总和，所以：

$$
\sum_{i=1}^{K}p_i=1
$$

因此 Softmax 的输出就是合法的概率分布。

## 5. 平移不变性与数值稳定

Softmax 有一个非常重要的性质：给所有 logits 同时加上同一个常数 $c$，结果不变。

$$
\operatorname{softmax}(z_i+c)=\frac{e^{z_i+c}}{\sum_{j=1}^{K}e^{z_j+c}}
$$

因为：

$$
e^{z_i+c}=e^c e^{z_i}
$$

所以：

$$
\operatorname{softmax}(z_i+c)=\frac{e^c e^{z_i}}{\sum_{j=1}^{K}e^c e^{z_j}}
$$

上下约去 $e^c$：

$$
\operatorname{softmax}(z_i+c)=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

实际计算时，常取：

$$
c=-\max(\mathbf{z})
$$

这样可以避免 $e^{z_i}$ 因为 $z_i$ 太大而溢出。稳定版 Softmax 写成：

$$
p_i=\frac{e^{z_i-\max(\mathbf{z})}}{\sum_{j=1}^{K}e^{z_j-\max(\mathbf{z})}}
$$

## 6. 和交叉熵损失的关系

多分类任务中，Softmax 常和交叉熵损失一起使用。

如果真实标签是 one-hot 向量：

$$
\mathbf{y}=[y_1,y_2,\dots,y_K]
$$

预测概率是：

$$
\mathbf{p}=\operatorname{softmax}(\mathbf{z})
$$

交叉熵损失是：

$$
\mathcal{L}=-\sum_{i=1}^{K}y_i\log p_i
$$

如果真实类别是第 $t$ 类，那么只有 $y_t=1$，其他 $y_i=0$，因此：

$$
\mathcal{L}=-\log p_t
$$

这表示：真实类别的预测概率越大，损失越小。

## 7. Softmax + 交叉熵的梯度结论

Softmax 和交叉熵组合后，有一个非常简洁的梯度结果：

$$
\frac{\partial \mathcal{L}}{\partial z_i}=p_i-y_i
$$

这个结果很重要。它说明每个类别 logit 的更新方向，就是预测概率和真实标签之间的差距。

如果某个错误类别的概率 $p_i$ 太大，而 $y_i=0$，那么：

$$
p_i-y_i>0
$$

梯度下降会压低这个类别的 logit。

如果真实类别 $t$ 的概率 $p_t$ 太小，而 $y_t=1$，那么：

$$
p_t-y_t<0
$$

梯度下降会抬高真实类别的 logit。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(logits):
    shifted = logits - np.max(logits)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits)

logits = np.array([1.2, 3.5, 0.8])
probs = softmax(logits)
target = np.array([0, 1, 0])
gradient = probs - target

print('logits:', logits)
print('softmax probabilities:', probs.round(4))
print('gradient p - y:', gradient.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
classes = ['class 1', 'class 2', 'class 3']

axes[0].bar(classes, probs, color=['#93C5FD', '#FCA5A5', '#86EFAC'])
axes[0].set_ylim(0, 1)
axes[0].set_title('Softmax Probabilities')
for i, p in enumerate(probs):
    axes[0].text(i, p + 0.03, f'{p:.2f}', ha='center')

axes[1].bar(classes, gradient, color=['#93C5FD', '#FCA5A5', '#86EFAC'])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Gradient: p - y')

plt.tight_layout()
plt.show()


## 8. 在 PyTorch 中的注意点

在 PyTorch 多分类训练中，通常使用：

```python
criterion = nn.CrossEntropyLoss()
```

`nn.CrossEntropyLoss()` 内部已经包含了 `log_softmax` 和负对数似然损失，所以模型最后一层通常直接输出 logits，不需要手动加 Softmax。

训练时：

$$
\text{model output}=\mathbf{z}
$$

推理时，如果需要查看概率，再单独计算：

$$
\mathbf{p}=\operatorname{softmax}(\mathbf{z})
$$